In [274]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
from sklearn.datasets import fetch_california_housing
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split

In [276]:
casa = pd.read_excel("./Datasets/01 ProvidenceClean.xlsx")
casas = casa.select_dtypes(include=[np.number])
casas.columns = casas.columns.str.strip()

In [278]:
matriz_correlacion = casas.corr()

In [280]:
caracteristicas_alta_correlacion = [(col1, col2, matriz_correlacion.loc[col1,col2])
                                   for col1 in matriz_correlacion.columns
                                   for col2 in matriz_correlacion.columns
                                   if col1 != col2 and abs(matriz_correlacion.loc[col1,col2])>0.8]

In [282]:
multicolinealidad_df = pd.DataFrame(caracteristicas_alta_correlacion, columns = ["Caracteristica 1","Caracteristica 2","Correlacion"])
print("?\nCaracteristicas altamente correlacionadas:\n", multicolinealidad_df)

?
Caracteristicas altamente correlacionadas:
    Caracteristica 1 Caracteristica 2  Correlacion
0       TOTAL VALUE              TAX     1.000000
1       TOTAL VALUE       GROSS AREA     0.800519
2       TOTAL VALUE      LIVING AREA     0.837120
3               TAX      TOTAL VALUE     1.000000
4               TAX       GROSS AREA     0.800518
5               TAX      LIVING AREA     0.837122
6        GROSS AREA      TOTAL VALUE     0.800519
7        GROSS AREA              TAX     0.800518
8        GROSS AREA      LIVING AREA     0.899775
9       LIVING AREA      TOTAL VALUE     0.837120
10      LIVING AREA              TAX     0.837122
11      LIVING AREA       GROSS AREA     0.899775


In [284]:
vif_df = pd.DataFrame()
vif_df["Caracteristica"]  = casas.columns 
vif_df["VIF"] = [variance_inflation_factor(casas.values, i) for i in range(casas.shape[1])]

In [286]:
print("\nFactor de Inflacion de Varianza (VIF) para cada caracteristica:\n",vif_df)


Factor de Inflacion de Varianza (VIF) para cada caracteristica:
    Caracteristica           VIF
0     TOTAL VALUE  3.187136e+08
1             TAX  3.186510e+08
2        LOT SQFT  1.007852e+01
3        YR BUILT  1.080647e+02
4      GROSS AREA  7.824469e+01
5     LIVING AREA  8.900994e+01
6          FLOORS  2.748655e+01
7           ROOMS  6.768609e+01
8        BEDROOMS  3.464224e+01
9       FULL BATH  1.079078e+01
10      HALF BATH  3.141452e+00
11        KITCHEN  7.517364e+01
12      FIREPLACE  3.167615e+00


In [288]:
X = casas.drop(columns=["TOTAL VALUE"])
y = casas["TOTAL VALUE"]

In [290]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [292]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaler = scaler.fit_transform(X_train)
X_test_scaler = scaler.transform(X_test)

In [294]:
from sklearn.linear_model import LinearRegression

In [296]:
model = LinearRegression()
model.fit(X_train_scaler, y_train)

LinearRegression()

In [298]:
y_prediccion = model.predict(X_test_scaler)

In [300]:
from sklearn.metrics import r2_score, mean_squared_error

In [302]:
r2 = r2_score(y_test,y_prediccion)
print(f"R: {r2:.4f}")

R: 1.0000


In [304]:
mse = mean_squared_error(y_test,y_prediccion)
print(f"Error Cuadratico Medio: {mse:.4f}")

Error Cuadratico Medio: 0.0005


In [306]:
rmse = mse**0.5
print(f"Raiz Error Cuadratico Medio: {rmse:.4f}")

Raiz Error Cuadratico Medio: 0.0229


In [308]:
print("Intercepto:", model.intercept_)
coeficientes_df = pd.DataFrame({"Caracteristica": X.columns, "Coeficiente": model.coef_})
print("\nCoeficientes de las caracteristicas:\n", coeficientes_df)

Intercepto: 392.36733527257053

Coeficientes de las caracteristicas:
    Caracteristica  Coeficiente
0             TAX    98.769084
1        LOT SQFT     0.000554
2        YR BUILT     0.000077
3      GROSS AREA     0.001379
4     LIVING AREA    -0.001643
5          FLOORS     0.000677
6           ROOMS    -0.000423
7        BEDROOMS    -0.000404
8       FULL BATH     0.000323
9       HALF BATH     0.000607
10        KITCHEN     0.000806
11      FIREPLACE     0.000522
